# 📝 [제 6편] 선형 회귀 (2) - 경사하강법

**학년/반/번호:** 2학년 ( )반 ( )번  **이름:** (           )

> 근거 자료: 박성호, 「Linear Regression (II) - 경사하강법(gradient descent algorithm)」 (ML_LEC_14)

---

## 🎯 학습 목표
1. 손실함수의 미분값(접선의 기울기)의 부호로 W 를 늘릴지 줄일지 판단할 수 있다.
2. 경사하강법의 갱신 식 W ← W − α × ∂E/∂W 를 손으로 몇 번 반복 계산할 수 있다.
3. 학습률 α 의 크기에 따라 학습이 어떻게 달라지는지 설명할 수 있다.

---

## 🧭 지난 시간 복습
* 제 5편: 손실함수 E(W) 는 아래로 볼록한 그릇 모양이고, **가장 낮은 곳의 W** 를 찾는 것이 학습입니다.
* 제 2편: 수치미분으로 어떤 점에서의 **접선의 기울기**를 계산할 수 있습니다.

> **오늘의 질문** 🤔 눈을 가리고 산 중턱에 서 있습니다. 골짜기 바닥으로 내려가려면 어떻게 해야 할까요?  
> → 발밑의 **경사(기울기)** 를 느끼고, **내려가는 방향**으로 한 발짝씩!

[⬅ 이전 : 제 5편 선형 회귀 (1) - 오차와 손실함수](https://colab.research.google.com/github/chenyenhee/ai-math-colab/blob/python/05_linear_regression_loss.ipynb)　|　[📚 목차](https://colab.research.google.com/github/chenyenhee/ai-math-colab/blob/python/00_index.ipynb)　|　[다음 : 제 7편 선형 회귀 (3) - 파이썬으로 직접 학습시키기 ➡](https://colab.research.google.com/github/chenyenhee/ai-math-colab/blob/python/07_linear_regression_python.ipynb)

> 💡 **처음 여는 경우** : 파일 → **Drive에 사본 저장** 을 누르고, 내 사본에서 공부하세요. (빈칸을 채우고 저장할 수 있습니다)
>
> 🧹 **다 본 뒤에는** : 수정 → **모든 출력 지우기** 후 저장하면 애니메이션 때문에 무거워진 사본이 다시 가벼워집니다.

## ⚙️ 0단계. 준비하기
아래 셀을 **가장 먼저 한 번** 실행하세요. (그래프 한글 글꼴과 애니메이션 설정)

In [ ]:
#@title ▶ 준비 셀 (먼저 실행하세요)
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm
from matplotlib import animation
from IPython.display import HTML, display
import urllib.request, os

font_path = 'NanumGothic.ttf'
try:
    if not os.path.exists(font_path):
        urllib.request.urlretrieve('https://raw.githubusercontent.com/google/fonts/main/ofl/nanumgothic/NanumGothic-Regular.ttf', font_path)
    fm.fontManager.addfont(font_path)
    plt.rc('font', family=fm.FontProperties(fname=font_path).get_name())
except Exception as e:
    print('글꼴을 내려받지 못했습니다(그래프의 한글이 깨질 수 있음):', e)
plt.rcParams['axes.unicode_minus'] = False
plt.rcParams['animation.embed_limit'] = 50
def show_anim(anim):
    # 애니메이션을 코랩 화면에 재생 버튼과 함께 보여 주기
    display(HTML(anim.to_jshtml()))
np.set_printoptions(precision=6, suppress=True)
print('✅ 준비 완료!')

## 📖 개념 다지기 1. 기울기의 부호가 알려 주는 방향

E(W) = (W − 3)² (W = 3 에서 최솟값) 를 예로 들어 봅시다.  E'(W) = 2(W − 3)

| 현재 W | 접선의 기울기 E'(W) | 최솟값은 어느 쪽? | W 를 어떻게? |
|:-:|:-:|:-:|:-:|
| W = 5 | 2(5 − 3) = 4 **(+)** | 왼쪽 | ( 늘린다 / 줄인다 ) |
| W = 1 | 2(1 − 3) = −4 **(−)** | 오른쪽 | ( 늘린다 / 줄인다 ) |
| W = 3 | 0 | 바로 여기! | 멈춘다 |

> 💡 **규칙**: 기울기가 양수이면 W 를 줄이고, 음수이면 W 를 늘린다.
> → 한 식으로 쓰면 **W 에서 기울기를 빼 주면 된다!**

## 📖 개념 다지기 2. 경사하강법 (Gradient Descent)

> **W ← W − α × ∂E(W)/∂W**

* ∂E(W)/∂W : 지금 위치에서 손실함수의 기울기 (편미분, 변수가 여럿이므로 ∂ 기호)
* **α (알파) : 학습률(learning rate)** - 한 번에 얼마나 크게 움직일지 정하는 작은 양수 (예: 0.1, 0.01, 0.0001)
* 이 계산을 기울기가 거의 0 이 될 때까지 **반복** 합니다.

바이어스 b 도 똑같이:  **b ← b − α × ∂E(W, b)/∂b**

### [연습 1] 손으로 경사하강법 3번 하기
E(W) = (W − 3)²,  E'(W) = 2(W − 3),  α = 0.1,  시작점 W = 0

| 반복 | 현재 W | 기울기 E'(W) | 새로운 W = W − 0.1 × E'(W) |
|:-:|:-:|:-:|:-:|
| 1 | 0 | 2(0 − 3) = −6 | 0 − 0.1 × (−6) = 0.6 |
| 2 | 0.6 | ( 　　 ) | ( 　　 ) |
| 3 | ( 　 ) | ( 　　 ) | ( 　　 ) |

> **[생각해 보기]** W 는 3 에 ( 가까워진다 / 멀어진다 ). 한 번에 움직이는 거리는 점점 ( 커진다 / 작아진다 ). 그 이유는?
> (                                                                                   )

In [ ]:
E  = lambda W : (W - 3)**2
dE = lambda W : 2*(W - 3)

W, alpha = 0.0, 0.1
for step in range(1, 11):
    g = dE(W)
    W_new = W - alpha * g
    print(f'{step:2d}번째 : W = {W:.4f},  기울기 = {g:8.4f},  새 W = {W_new:.4f},  E = {E(W_new):.6f}')
    W = W_new

## 🎬 애니메이션 1. 공이 그릇 바닥으로 굴러간다

E(W) = (W − 3)² 위에서 W = 0 에서 출발하여 경사하강법으로 움직입니다. 빨간 선은 그 점에서의 **접선**입니다.

In [ ]:
#@title 🎬 애니메이션 1 실행 (경사하강법)
alpha = 0.1
hist = [0.0]
for _ in range(25):
    hist.append(hist[-1] - alpha * dE(hist[-1]))
frames = []
for k in range(len(hist)):
    frames += [k] * 3

fig, ax = plt.subplots(figsize=(8, 5.5))
Ws = np.linspace(-1, 7, 200)
def draw(f):
    k = frames[f]; W = hist[k]; g = dE(W)
    ax.clear()
    ax.plot(Ws, E(Ws), color='purple', lw=2, label='E(W) = (W − 3)²')
    ax.plot(hist[:k+1], [E(w) for w in hist[:k+1]], 'o-', color='gray', ms=4, alpha=0.6)
    tl = np.linspace(W - 1.5, W + 1.5, 10)
    ax.plot(tl, E(W) + g*(tl - W), 'r-', lw=2, label=f'접선 기울기 = {g:.3f}')
    ax.plot([W], [E(W)], 'o', color='crimson', ms=14, mec='k', zorder=5)
    if k < len(hist) - 1:
        ax.annotate('', xy=(hist[k+1], -0.6), xytext=(W, -0.6), arrowprops=dict(arrowstyle='->', color='green', lw=2.5))
    ax.set_ylim(-1.5, 16); ax.set_xlim(-1, 7); ax.grid(alpha=0.3); ax.legend(loc='upper right')
    ax.set_xlabel('W'); ax.set_ylabel('E(W)')
    ax.set_title(f'{k}번째 반복 :  W = {W:.4f}   (α = {alpha})', fontsize=14)
anim = animation.FuncAnimation(fig, draw, frames=len(frames), interval=120)
plt.close(fig)
show_anim(anim)

> **[관찰 1]** ① 처음에는 크게, 나중에는 작게 움직인다. 그 이유는 ( 　　　　　 ) 가 점점 0 에 가까워지기 때문이다.
> ② 초록 화살표는 기울기가 음수일 때 ( 오른쪽 / 왼쪽 ) 을 가리킨다.

## 🎬 애니메이션 2. 학습률 α 가 너무 작거나 크면?

같은 출발점(W = 0)에서 학습률만 바꿔서 비교합니다.
* α = 0.02 (너무 작음),  α = 0.3 (적당함),  α = 0.95 (큼),  α = 1.05 (너무 큼)

In [ ]:
#@title 🎬 애니메이션 2 실행 (학습률 비교)
alphas = [0.02, 0.3, 0.95, 1.05]
titles = ['너무 작음', '적당함', '큼 (출렁임)', '너무 큼 (발산!)']
H = []
for a in alphas:
    h = [0.0]
    for _ in range(20):
        h.append(h[-1] - a * dE(h[-1]))
    H.append(h)

fig, axes = plt.subplots(1, 4, figsize=(16, 4.2))
Ws = np.linspace(-3, 9, 200)
def draw(k):
    for ax, a, h, tt in zip(axes, alphas, H, titles):
        ax.clear()
        ax.plot(Ws, E(Ws), color='purple', lw=1.5)
        hh = h[:k+1]
        ax.plot(hh, [E(w) for w in hh], 'o-', color='crimson', ms=4, lw=1)
        ax.plot([hh[-1]], [E(hh[-1])], 'o', color='crimson', ms=11, mec='k')
        ax.set_xlim(-3, 9); ax.set_ylim(-1, 40)
        ax.set_title(f'α = {a}  ({tt})\n{k}번째 W = {hh[-1]:.2f}', fontsize=12)
        ax.grid(alpha=0.3)
anim = animation.FuncAnimation(fig, draw, frames=21, interval=350)
plt.close(fig)
show_anim(anim)

> **[관찰 2]** 표를 완성하세요.
>
> | α | 20번 반복 뒤 W | 결과 |
> |:-:|:-:|:-|
> | 0.02 | ( 　 ) | 바닥에 ( 아직 못 갔다 / 도착했다 ) |
> | 0.3 | ( 　 ) | |
> | 1.05 | ( 　 ) | 바닥에서 점점 ( 　　　 ) |
>
> 💡 학습률은 사람이 정해 주는 값(**하이퍼파라미터**)입니다. 보통 10<sup>−2</sup>, 10<sup>−3</sup>, 10<sup>−4</sup> 등을 바꿔 가며 실험해서 정합니다.

## 📖 개념 다지기 3. W 와 b 를 함께 내려가기

실제 선형 회귀에서는 W 와 b 를 **동시에** 갱신합니다.

　W ← W − α × ∂E/∂W,　　b ← b − α × ∂E/∂b

데이터 x = (1, 2, 3, 4, 5), t = (2, 3, 4, 5, 6) 에 대해 E(W, b) 의 등고선 지도 위에서 (W, b) 가 움직이는 길을 봅시다.
(정답은 t = x + 1 이므로 W = 1, b = 1 에 도착해야 합니다)

In [ ]:
#@title 🎬 애니메이션 3 실행 (등고선 위의 경사하강법)
xd = np.array([1, 2, 3, 4, 5.]); td = np.array([2, 3, 4, 5, 6.])
Ewb = lambda W, b : np.mean((td - (W*xd + b))**2)
def grad(W, b, h=1e-4):          # 제 2편의 수치미분 (중앙차분)
    return np.array([(Ewb(W + h, b) - Ewb(W - h, b)) / (2*h), (Ewb(W, b + h) - Ewb(W, b - h)) / (2*h)])

p = np.array([-1.0, 4.0]); alpha = 0.05; path = [p.copy()]
for _ in range(300):
    p = p - alpha * grad(*p); path.append(p.copy())
path = np.array(path)

Wg, bg = np.meshgrid(np.linspace(-1.5, 2.5, 120), np.linspace(-1, 5, 120))
Eg = np.vectorize(Ewb)(Wg, bg)
idx = list(range(0, 30)) + list(range(30, 301, 15)) + [300]*6
fig, (axL, axR) = plt.subplots(1, 2, figsize=(11, 4.3))
axL.contour(Wg, bg, Eg, levels=np.logspace(-2, 2.3, 16), cmap='viridis')
axL.plot(1, 1, 'k*', ms=15, label='정답 (W=1, b=1)')
line, = axL.plot([], [], 'o-', color='crimson', ms=3)
axL.set_xlabel('W'); axL.set_ylabel('b'); axL.legend(loc='upper right')
def draw(f):
    k = idx[f]
    line.set_data(path[:k+1, 0], path[:k+1, 1])
    axL.set_title(f'{k}번째 :  W = {path[k,0]:.3f},  b = {path[k,1]:.3f}', fontsize=13)
    axR.clear()
    axR.scatter(xd, td, s=70, color='royalblue', zorder=5)
    xs = np.linspace(0, 6, 10)
    axR.plot(xs, path[k,0]*xs + path[k,1], color='crimson', lw=2)
    axR.set_xlim(0, 6); axR.set_ylim(-2, 10); axR.grid(alpha=0.3)
    axR.set_title(f'직선 y = {path[k,0]:.2f}x + {path[k,1]:.2f},  E = {Ewb(*path[k]):.4f}', fontsize=13)
    return line,
anim = animation.FuncAnimation(fig, draw, frames=len(idx), interval=120)
plt.close(fig)
show_anim(anim)

> **[관찰 3]** 처음에는 b 가 크게 변하지 않고 주로 ( W / b ) 가 먼저 빠르게 변한 뒤, 골짜기를 따라 천천히 정답으로 다가간다.
> 왼쪽 지도의 점이 정답 ★ 에 가까워질수록 오른쪽 직선은 ( 　　　　　 ).

## 📖 개념 다지기 4. 학습 과정 한눈에 보기 (순서도)

```
 ① 학습 데이터 입력 (x, t)
        ↓
 ② y = Wx + b 계산 → 손실함수 E(W, b) 계산
        ↓
 ③ E 가 충분히 작은가? ── 예 ──→ 학습 종료 (최종 W, b)
        │ 아니오
        ↓
 ④ W ← W − α × ∂E/∂W,  b ← b − α × ∂E/∂b  로 갱신 → ② 로 돌아가 반복
```

## ✏️ 연습 문제

> **[문제 1]** E(W) = (W + 1)² + 2 일 때 E'(W) = ( 　　　　 ). α = 0.25, 시작점 W = 3 에서 경사하강법을 2번 하면 W = ( 　 ) → ( 　 ) 이다.

> **[문제 2]** 학습률 α = 1 로 [문제 1] 을 하면 어떻게 될까요? 예상한 뒤 코드로 확인하세요.
> 예상: (                                         )

아래 셀의 `______` 를 채워 실행하세요.

In [ ]:
E1  = lambda W : (W + 1)**2 + 2
dE1 = lambda W : ______

for alpha in [0.25, 1.0]:
    W = 3.0
    print(f'α = {alpha}')
    for step in range(1, 6):
        W = W - alpha * dE1(W)
        print(f'   {step}번째 W = {W:.4f}')

In [ ]:
#@title 🔑 정답 확인 (먼저 풀어 본 뒤에 실행하세요) { display-mode: "form" }
print("문제 1 : E'(W) = 2(W + 1)")
W = 3.0
for k in range(2):
    W = W - 0.25 * 2*(W + 1); print(f'   {k+1}번째 W = {W}')
print('문제 2 : α = 1 이면 W = 3 → −5 → 3 → −5 … 로 최솟값(W = −1) 양쪽을 왔다 갔다 하며 수렴하지 않는다.')

## 📌 정리

| 핵심 | 내용 |
|:---|:---|
| 기울기 부호 | 양수 → W 줄이기, 음수 → W 늘리기 |
| 경사하강법 | W ← W − α × ∂E/∂W,  b ← b − α × ∂E/∂b  를 반복 |
| 학습률 α | 너무 작으면 느리고, 너무 크면 발산한다 |
| 기울기 계산 | 제 2편의 수치미분 함수로 계산한다 |

### 🏆 도전 과제
경사하강법은 "지금 서 있는 곳의 기울기" 만 보고 내려갑니다. 만약 그릇이 아니라 **움푹한 곳이 여러 개인 산** 이라면 어떤 문제가 생길까요?
(                                                                                                                            )

---

[⬅ 이전 : 제 5편 선형 회귀 (1) - 오차와 손실함수](https://colab.research.google.com/github/chenyenhee/ai-math-colab/blob/python/05_linear_regression_loss.ipynb)　|　[📚 목차](https://colab.research.google.com/github/chenyenhee/ai-math-colab/blob/python/00_index.ipynb)　|　[다음 : 제 7편 선형 회귀 (3) - 파이썬으로 직접 학습시키기 ➡](https://colab.research.google.com/github/chenyenhee/ai-math-colab/blob/python/07_linear_regression_python.ipynb)